## Arguments

In [4]:
VIDEOS_DIR = "C:/Users/chany/Downloads/test"
FRAME_INTERVAL = 30
MULTITHREAD = True
SERVER_MACHINE = "chan@223.130.138.39"
SERVER_DATA_PATH = "/home/chan/data"

## Setup

In [5]:
%load_ext autoreload
%autoreload 2

import os
import sys
import json

dir_base = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(dir_base)

from utils.save_every_nth_frames_from_video import save_frames_from_videos

OUTPUT_DIR = os.path.realpath(os.path.join(VIDEOS_DIR, "data_"+os.path.basename(VIDEOS_DIR)))
VIDEOS_DIR = os.path.realpath(VIDEOS_DIR)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Videos -> Images

In [6]:
save_frames_from_videos(
    videos_folder_path=VIDEOS_DIR,
    frame_interval=FRAME_INTERVAL,
    multithread=MULTITHREAD
)

Press 'q' to stop the process.
Process terminated by user.


## 2. SFM

In [7]:
os.system(f"""{os.path.join(dir_base, "reality_capture", "RC_SFM.BAT")} {VIDEOS_DIR}""")

0

## 3. Modify registration

In [8]:
TRANSFORMS_PATH_NEW = os.path.join(OUTPUT_DIR, "transforms.json")

os.replace(
    os.path.join(OUTPUT_DIR, "images", "transforms.json"),
    TRANSFORMS_PATH_NEW
)

# Load the JSON data from the file
with open(TRANSFORMS_PATH_NEW, "r") as file:
    data = json.load(file)

# Update the file_path values
for frame in data["frames"]:
    # Extract the numeric part of the filename and reformat the path
    filename = os.path.basename(frame["file_path"])
    number_part = filename.split(".")[0]  # Get the part before ".jpg"
    new_file_path = f"images/{number_part.zfill(5)}.jpg"  # Pad with leading zeros
    frame["file_path"] = new_file_path

data["ply_file_path"] = "sparse_pc.ply"

# Save the modified data back to the JSON file
with open(TRANSFORMS_PATH_NEW, "w") as file:
    json.dump(data, file, indent=4)

print("File paths have been updated successfully.")

File paths have been updated successfully.


## 4. Transfer to server

In [13]:
# command = f"scp -vr {OUTPUT_DIR} {SERVER_MACHINE}:{SERVER_DATA_PATH}"
# print(command)
# os.system(command)

command = f"scp -vr {os.path.join(dir_base, 'scripts', 'server-scripts')} {SERVER_MACHINE}:/home/chan/"
print(command)
os.system(command)

scp -vr c:\Users\chany\Documents\SWAROBO\reconstruction\scripts\server-scripts chan@223.130.138.39:/home/chan/


0

## 5. Start training process

In [14]:
SERVER_DATA_ADDED_PATH = SERVER_DATA_PATH+"/"+os.path.basename(OUTPUT_DIR)
command = f"ssh {SERVER_MACHINE} bash /home/chan/server-scripts/run.bash {SERVER_DATA_ADDED_PATH}"
print(command)
os.system(command)

ssh chan@223.130.138.39 bash /home/chan/server-scripts/run.bash /home/chan/data/data_test


0